> **Archived research notebook.** Retained for reproducibility and comparisons; it may require the historical code/dependencies. Use the [current notebook catalog](https://github.com/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/README.md) for new runs.


# Qwen3.5-0.8B + FlyEmbedding-v3 — identity-preserving residual adapter

This keeps the original Qwen embedding and LM head unchanged.

`e_v3 = e_qwen + alpha * Fly(e_qwen)`

`alpha` starts at exactly zero, so the notebook verifies exact embedding identity, 100% top-1 agreement, and identical deterministic generation before training.


In [1]:
#@title 1. Update repository, install dependencies, and preflight
import pathlib, subprocess, sys, importlib
REPO_DIR=pathlib.Path('/content/TinyCeNN-LM')
if REPO_DIR.exists():
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin'],check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard','origin/main'],check=True)
else:
    subprocess.run(['git','clone','https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO_DIR)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-U','transformers','accelerate','huggingface_hub','safetensors','ipywidgets','pandas'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO_DIR)],check=True)
SRC_DIR=REPO_DIR/'src'
if str(SRC_DIR) not in sys.path: sys.path.insert(0,str(SRC_DIR))
for _name in list(sys.modules):
    if _name == 'tinycenn_lm' or _name.startswith('tinycenn_lm.'): del sys.modules[_name]
importlib.invalidate_caches()
for p in [REPO_DIR/'scripts'/'run_qwen35_flyembedding_v3.py', REPO_DIR/'src'/'tinycenn_lm'/'qwen35_flyembedding_v3.py']:
    subprocess.run([sys.executable,'-m','py_compile',str(p)],check=True)
from tinycenn_lm.qwen35_flyembedding_v3 import FlyEmbeddingV3Config, install_fly_embedding_v3
print('✓ FlyEmbedding-v3 preflight OK')


✓ FlyEmbedding-v3 preflight OK


In [2]:
#@title 2. Configuration
BASE_MODEL='Qwen/Qwen3.5-0.8B' #@param {type:'string'}
RUN_MODE='quick' #@param ['quick','strong']
SEQ_LEN=128 #@param {type:'integer'}
FLY_NODES=256 #@param {type:'integer'}
GRAPH_STEPS=1 #@param {type:'integer'}
GRAPH_MIX_INIT=0.05 #@param {type:'number'}
MAX_RESIDUAL_SCALE=0.05 #@param {type:'number'}
LR_CORE=0.0002 #@param {type:'number'}
LR_GATE=0.0005 #@param {type:'number'}
OUTPUT_DIR=REPO_DIR/'results'/'flyembedding_v3_qwen35_08b'
print('Base:',BASE_MODEL)
print('Original Qwen embedding preserved: YES')
print('Original Qwen lm_head preserved: YES')
print('Residual gate starts at zero: YES')


Base: Qwen/Qwen3.5-0.8B
Original Qwen embedding preserved: YES
Original Qwen lm_head preserved: YES
Residual gate starts at zero: YES


In [3]:
#@title 3. Run FlyEmbedding-v3 — live output
import subprocess, sys
cmd=[sys.executable,'-u',str(REPO_DIR/'scripts'/'run_qwen35_flyembedding_v3.py'),
     '--base-model',BASE_MODEL,'--run-mode',RUN_MODE,'--seq-len',str(SEQ_LEN),
     '--fly-nodes',str(FLY_NODES),'--graph-steps',str(GRAPH_STEPS),
     '--graph-mix-init',str(GRAPH_MIX_INIT),'--max-residual-scale',str(MAX_RESIDUAL_SCALE),
     '--lr-core',str(LR_CORE),'--lr-gate',str(LR_GATE),'--output-dir',str(OUTPUT_DIR)]
print('='*100); print(' '.join(cmd)); print('='*100)
p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in iter(p.stdout.readline,''): print(line,end='',flush=True)
rc=p.wait(); print('\nFinished, exit code',rc)
if rc: raise subprocess.CalledProcessError(rc,cmd)


/usr/bin/python3 -u /content/TinyCeNN-LM/scripts/run_qwen35_flyembedding_v3.py --base-model Qwen/Qwen3.5-0.8B --run-mode quick --seq-len 128 --fly-nodes 256 --graph-steps 1 --graph-mix-init 0.05 --max-residual-scale 0.05 --lr-core 0.0002 --lr-gate 0.0005 --output-dir /content/TinyCeNN-LM/results/flyembedding_v3_qwen35_08b
DEVICE cpu | dtype torch.float32
Loading Qwen teacher...

Loading weights: 100%|██████████| 320/320 [00:01<00:00, 232.66it/s]
Loading Qwen + FlyEmbedding-v3 student...

Loading weights: 100%|██████████| 320/320 [00:01<00:00, 180.99it/s]
IDENTITY EMBEDDING CHECK {"exact_identity": true, "max_abs_embedding_error": 0.0}
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not instal

In [4]:
#@title 4. Results
import json, pandas as pd
from IPython.display import display
report=json.loads((OUTPUT_DIR/'report.json').read_text())
hist=pd.read_csv(OUTPUT_DIR/'training_history.csv')
print('Architecture:',report['architecture'])
print('Identity:',report['identity_embedding_check'])
print('Initial deterministic generation exact:',report['initial_generation_exact'])
print('Parameter stats:\n',json.dumps(report['parameter_stats'],indent=2))
print('Initial probe:\n',json.dumps(report['initial_probe'],indent=2))
print('Best probe:\n',json.dumps(report['best_probe'],indent=2))
print('Final probe:\n',json.dumps(report['final_probe'],indent=2))
print('QUALITY GATE:',report['quality_gate_passed'])
display(hist.tail(20))
for x in report['generation_samples']:
    print('\nUSER:',x['prompt'])
    print('QWEN:',x['qwen_reply'])
    print('FLY :',x['fly_reply'])
    print('exact=',x['exact_token_match'],'| prefix=',x['matching_prefix_tokens'],'| jaccard=',round(x['token_jaccard'],3),'| passed=',x['passed'])


Architecture: Qwen3.5-0.8B + identity-preserving FlyEmbedding-v3 residual adapter
Identity: {'exact_identity': True, 'max_abs_embedding_error': 0.0}
Initial deterministic generation exact: True
Parameter stats:
 {
  "base_embedding_params_preserved": 254279680,
  "adapter_trainable_params": 525314,
  "adapter_overhead_vs_embedding_pct": 0.20658905973139496,
  "fly_nodes": 256,
  "graph_steps": 1,
  "residual_scale": 0.006239418871700764,
  "graph_mix": 0.0441293865442276
}
Initial probe:
 {
  "teacher_ce": 2.583987534046173,
  "student_ce": 2.583987534046173,
  "ce_gap": 0.0,
  "teacher_kl": -2.075651858723937e-09,
  "embedding_relative_mse": 0.0,
  "top1_logit_agreement": 1.0,
  "max_abs_logit_error": 0.0
}
Best probe:
 {
  "teacher_ce": 2.583987534046173,
  "student_ce": 2.464639335870743,
  "ce_gap": -0.1193481981754303,
  "teacher_kl": 0.014183854742441326,
  "embedding_relative_mse": 0.0036014381039422005,
  "top1_logit_agreement": 0.955078125,
  "max_abs_logit_error": 4.910690307

,update,loss,ce,kl,embedding_relative_mse,residual_scale,graph_mix
280,281,0.490355,2.373485,0.021748,0.013046,0.008415,0.041839
281,282,0.485516,2.339706,0.025220,0.012213,0.008441,0.041811
282,283,0.629092,3.038240,0.031132,0.013827,0.008467,0.041783
283,284,0.476670,2.308714,0.020574,0.012914,0.008494,0.041755
284,285,0.431340,2.079331,0.021478,0.012936,0.008519,0.041727
285,286,0.532608,2.577555,0.024003,0.013475,0.008543,0.041702
286,287,0.581433,2.809020,0.028348,0.013102,0.008568,0.041676
287,288,0.485054,2.358331,0.016744,0.016706,0.008592,0.041650
288,289,0.449749,2.151444,0.027214,0.015659,0.008617,0.041624
289,290,0.619997,3.011740,0.024839,0.013730,0.008641,0.041598



USER: Explain in two sentences why the sky is blue.
QWEN: The sky appears blue because sunlight scatters off the Earth's atmosphere, with shorter wavelengths of blue light being scattered more effectively than longer ones, making the sky look blue to our eyes.
FLY : The sky appears blue because sunlight scatters off the Earth's atmosphere, with shorter wavelengths of blue light being scattered more effectively than longer ones, while the longer wavelengths of red and orange light travel further and reach our eyes.
exact= False | prefix= 28 | jaccard= 0.762 | passed= True

USER: What is 17 + 25? Give only the answer.
QWEN: 42
FLY : 42
exact= True | prefix= 5 | jaccard= 1.0 | passed= True

USER: Write one short sentence about Vienna.
QWEN: Vienna is the capital of Austria and the largest city in Europe, known for its historic architecture, vibrant culture, and stunning natural landscapes.
FLY : Vienna is the capital of Austria, known for its historic architecture, vibrant culture, and s

In [5]:
#@title 5. Reload adapter and compare interactively
import torch, ipywidgets as widgets
from IPython.display import display, clear_output
from transformers import AutoModelForCausalLM, AutoTokenizer
from tinycenn_lm.qwen35_flyembedding_v3 import FlyEmbeddingV3Config, install_fly_embedding_v3
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype=torch.bfloat16 if device.type=='cuda' and torch.cuda.is_bf16_supported() else (torch.float16 if device.type=='cuda' else torch.float32)
tok=AutoTokenizer.from_pretrained(BASE_MODEL,use_fast=True)
if tok.pad_token_id is None: tok.pad_token=tok.eos_token
qwen=AutoModelForCausalLM.from_pretrained(BASE_MODEL,dtype=dtype,low_cpu_mem_usage=True).to(device).eval()
fly=AutoModelForCausalLM.from_pretrained(BASE_MODEL,dtype=dtype,low_cpu_mem_usage=True).to(device).eval()
def adj(n):
    a=torch.zeros(n,n,dtype=torch.float32)
    for i in range(n):
        a[i,i]=1
        for s in (1,3,7,17):
            a[i,(i+s)%n]=1; a[i,(i-s)%n]=1
    return (a/a.sum(-1,keepdim=True).clamp_min(1)).to(device)
ckpt=torch.load(OUTPUT_DIR/'fly_embedding_v3_adapter.pt',map_location='cpu')
cfg=FlyEmbeddingV3Config(**ckpt['config'])
install_fly_embedding_v3(fly,cfg,adj(cfg.fly_nodes))
fly.fly_embedding_v3_core.load_state_dict(ckpt['fly_embedding_v3_core'],strict=True)
fly.eval()
def answer(model,prompt):
    text=tok.apply_chat_template([{'role':'user','content':prompt}],tokenize=False,add_generation_prompt=True)
    enc=tok(text,return_tensors='pt').to(device)
    with torch.no_grad():
        y=model.generate(**enc,max_new_tokens=128,do_sample=False,use_cache=True,pad_token_id=tok.eos_token_id)
    return tok.decode(y[0,enc.input_ids.shape[1]:],skip_special_tokens=True).strip()
prompt_box=widgets.Textarea(value='Explain why residual adapters can preserve a pretrained model.',description='Prompt:',layout=widgets.Layout(width='100%',height='90px'))
button=widgets.Button(description='Compare',button_style='primary')
out=widgets.Output()
def run(_):
    q=prompt_box.value.strip()
    if not q:return
    with out:
        clear_output()
        print('QWEN:\n',answer(qwen,q)); print('\n'+'-'*80+'\n'); print('FLY-v3:\n',answer(fly,q))
button.on_click(run)
display(prompt_box,button,out)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Textarea(value='Explain why residual adapters can preserve a pretrained model.', description='Prompt:', layout…

Button(button_style='primary', description='Compare', style=ButtonStyle())

Output()